# 04. То же самое через nn.Module и optim.SGD

Три предыдущих ноутбука собирали сеть из тензоров: параметры объявляли руками,
прямой проход писали выражением, шаг спуска писали циклом. Здесь та же сеть
собирается штатными средствами PyTorch, и мы построчно сопоставляем, **что
именно библиотека сделала за нас**.

Смысл ноутбука не в том, чтобы показать "как надо писать на самом деле".
Смысл в том, чтобы за каждым вызовом `nn.Linear`, `criterion`, `optimizer.step()`
вы видели конкретную формулу из ноутбуков 02-03.

## Карта соответствий

| Вручную (ноутбуки 02-03) | Штатно (этот ноутбук) | Что внутри |
|---|---|---|
| `w = randn(d,h)/sqrt(d); w.requires_grad_()` | `nn.Linear(d, h)` | та же инициализация по $1/\sqrt{n_{in}}$, плюс регистрация параметра в модуле |
| `z = x @ w + b` | `layer(x)` | то же матричное умножение |
| `z.clamp(min=0)` | `F.relu(z)` | то же |
| `nll(log_softmax(z), y)` | `nn.CrossEntropyLoss()(z, y)` | тот же устойчивый log_softmax плюс NLL, объединенные в одну операцию |
| `loss.backward()` | `loss.backward()` | **одно и то же**, autograd мы использовали всегда |
| `p -= lr*p.grad` под `no_grad()` | `optimizer.step()` | то же вычитание, плюс momentum и прочие схемы |
| `p.grad.zero_()` | `optimizer.zero_grad()` | то же обнуление |
| `iterate_minibatches(...)` | `DataLoader` | то же перемешивание и нарезка, плюс многопоточная загрузка |

Содержательно нового в правом столбце нет ничего. Есть удобство: параметры
не надо перечислять вручную, оптимизатор умеет больше схем, `DataLoader`
тянет данные с диска параллельно. На MNIST из памяти это не нужно, на
ImageNet - необходимо.

In [ ]:
import math
from pathlib import Path
import gzip
import pickle

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import matplotlib.pyplot as plt

torch.manual_seed(0)
print("torch", torch.__version__)

In [ ]:
DATA_PATH = Path("data")
PATH = DATA_PATH / "mnist"
PATH.mkdir(parents=True, exist_ok=True)
FILENAME = "mnist.pkl.gz"

if not (PATH / FILENAME).exists():
    import requests
    URL = "https://github.com/pytorch/tutorials/raw/main/_static/"
    (PATH / FILENAME).open("wb").write(requests.get(URL + FILENAME, timeout=60).content)

with gzip.open((PATH / FILENAME).as_posix(), "rb") as f:
    (x_train, y_train), (x_valid, y_valid), _ = pickle.load(f, encoding="latin-1")

x_train = torch.tensor(x_train, dtype=torch.float32)
y_train = torch.tensor(y_train, dtype=torch.int64)
x_valid = torch.tensor(x_valid, dtype=torch.float32)
y_valid = torch.tensor(y_valid, dtype=torch.int64)

n, d = x_train.shape
n_hidden, num_classes = 128, 10
print(f"train {tuple(x_train.shape)}, valid {tuple(x_valid.shape)}")

---
## 1. Проверка 1: nn.Linear - это ровно `x @ W + b`

Единственная тонкость - **транспонирование**. У нас было `W` формы
$(n_{in}, n_{out})$ и `z = x @ W`. У `nn.Linear` вес хранится как
$(n_{out}, n_{in})$, а применяется как `x @ W.T`. Результат тот же, разница
только в том, как матрица лежит в памяти.

Проверим численно: возьмем `nn.Linear`, вытащим из него вес и смещение,
посчитаем выход своей формулой и сравним.

In [ ]:
layer = nn.Linear(d, n_hidden)

print("наше соглашение:       W (n_in, n_out) =", (d, n_hidden))
print("соглашение nn.Linear:  weight.shape    =", tuple(layer.weight.shape))
print("                       bias.shape      =", tuple(layer.bias.shape))

xb = x_train[:64]

with torch.no_grad():
    out_torch = layer(xb)                              # штатно
    out_manual = xb @ layer.weight.T + layer.bias      # наша формула

print(f"\nmax |nn.Linear(x) - (x @ W.T + b)| = "
      f"{(out_torch - out_manual).abs().max().item():.3e}")
print("совпало:", torch.allclose(out_torch, out_manual, atol=1e-6))

### Какую инициализацию выбирает nn.Linear

В ноутбуке 02 мы брали $W \sim \mathcal{N}(0,1)/\sqrt{n_{in}}$ и обосновывали
это ростом дисперсии суммы. `nn.Linear` по умолчанию делает другое, но с той
же целью: берет равномерное распределение на $[-1/\sqrt{n_{in}}, 1/\sqrt{n_{in}}]$
(инициализация Кайминга в варианте для равномерного распределения).

Распределения разные, **масштаб тот же** $1/\sqrt{n_{in}}$. Сравним
стандартные отклонения: у равномерного на $[-a, a]$ это $a/\sqrt{3}$.

In [ ]:
a = 1 / math.sqrt(d)
print(f"n_in = {d},  1/sqrt(n_in) = {a:.5f}")
print()
print(f"наша инициализация,  std = 1/sqrt(n_in)   = {a:.5f}")
print(f"nn.Linear (равномерная), std = a/sqrt(3)  = {a / math.sqrt(3):.5f}")
print(f"фактический std weight у nn.Linear        = {layer.weight.std().item():.5f}")
print(f"фактический min/max                       = "
      f"{layer.weight.min().item():.5f} / {layer.weight.max().item():.5f}")
print(f"\nграница a = {a:.5f} - веса лежат ровно в [-a, a]")
print("смещение nn.Linear тоже инициализирует не нулем, а равномерно в том же диапазоне")

---
## 2. Проверка 2: CrossEntropyLoss - это наш `nll(log_softmax(z), y)`

Важная ловушка, на которой спотыкаются постоянно: `nn.CrossEntropyLoss`
принимает **логиты**, а не вероятности. Он сам внутри применяет `log_softmax`.
Если подать ему `softmax(z)`, получится softmax от softmax - модель будет
учиться, но заметно хуже, и ошибку легко не заметить.

Проверим эквивалентность, а потом покажем, во что превращается ошибка
при двойном softmax.

In [ ]:
def log_softmax_manual(z):
    m = z.max(dim=-1, keepdim=True).values
    zs = z - m
    return zs - zs.exp().sum(dim=-1, keepdim=True).log()


def nll_manual(log_probs, y):
    return -log_probs[range(y.shape[0]), y].mean()


def softmax_manual(z):
    zs = z - z.max(dim=-1, keepdim=True).values
    e = zs.exp()
    return e / e.sum(dim=-1, keepdim=True)


yb = y_train[:64]
logits = torch.randn(64, num_classes)

criterion = nn.CrossEntropyLoss()

loss_torch = criterion(logits, yb)
loss_manual = nll_manual(log_softmax_manual(logits), yb)
loss_wrong = criterion(softmax_manual(logits), yb)   # типовая ошибка

print(f"nn.CrossEntropyLoss(логиты):       {loss_torch.item():.6f}")
print(f"наш nll(log_softmax(логиты)):      {loss_manual.item():.6f}")
print(f"совпало: {torch.allclose(loss_torch, loss_manual, atol=1e-6)}")
print()
print(f"ОШИБКА - подали softmax вместо логитов: {loss_wrong.item():.6f}")
print("значение другое, и такая модель обучается хуже, хотя падение ошибки видно")

### И наша главная формула градиента - тоже внутри

Проверим, что `(softmax(z) - Y)/B` - это в точности то, что `backward()`
штатного `CrossEntropyLoss` кладет в градиент по логитам.

In [ ]:
z_leaf = logits.clone().requires_grad_()
criterion(z_leaf, yb).backward()

Y = torch.zeros(64, num_classes)
Y[range(64), yb] = 1.0
g_manual = (softmax_manual(logits) - Y) / 64

print(f"max |наша формула - градиент от CrossEntropyLoss| = "
      f"{(g_manual - z_leaf.grad).abs().max().item():.3e}")
print("совпало:", torch.allclose(g_manual, z_leaf.grad, atol=1e-6))
print("\nто есть dL/dz = (softmax(z) - Y) / B - это и есть то, что считает PyTorch")

---
## 3. Та же сеть как nn.Module

`nn.Module` - это контейнер. Он делает три вещи:

1. **собирает параметры**: любой `nn.Parameter` или вложенный модуль,
   присвоенный в `__init__`, автоматически попадает в `self.parameters()`.
   Нам больше не надо перечислять `w1, b1, w2, b2` руками - именно это
   становится важно, когда слоев не два, а пятьдесят;
2. **задает прямой проход** методом `forward`; вызов `net(x)` идет в него;
3. **переключает режимы** `train()`/`eval()` - для dropout и batch norm
   поведение при обучении и при оценке различается. В нашей сети таких
   слоев нет, поэтому разницы не будет, но вызывать правильный режим -
   хорошая привычка.

In [ ]:
class MLP(nn.Module):
    def __init__(self, n_in=784, n_hidden=128, n_out=10):
        super().__init__()
        self.fc1 = nn.Linear(n_in, n_hidden)
        self.fc2 = nn.Linear(n_hidden, n_out)

    def forward(self, x):
        x = F.relu(self.fc1(x))
        return self.fc2(x)      # возвращаем ЛОГИТЫ: softmax сделает CrossEntropyLoss


torch.manual_seed(0)
net = MLP(d, n_hidden, num_classes)

print(net)
print("\nпараметры, которые модуль собрал сам:")
total = 0
for name, p in net.named_parameters():
    print(f"  {name:<12} {str(tuple(p.shape)):<14} {p.numel():>7} шт.")
    total += p.numel()
print(f"\nвсего: {total}")
print("в ноутбуке 03 вручную было: 101770")

---
## 4. optimizer.step() - это наш цикл вычитания

`optim.SGD(net.parameters(), lr=lr)` с параметрами по умолчанию
(`momentum=0`) делает ровно то же, что мы писали руками:

```python
with torch.no_grad():
    for p in params:
        p -= lr * p.grad
```

Проверим это: сделаем один шаг обеими способами из одной точки и сравним
получившиеся веса.

In [ ]:
torch.manual_seed(0)
net_a = MLP(d, n_hidden, num_classes)

torch.manual_seed(0)
net_b = MLP(d, n_hidden, num_classes)   # та же инициализация

lr = 0.1
xb, yb = x_train[:64], y_train[:64]

# вариант A: штатный оптимизатор
opt = optim.SGD(net_a.parameters(), lr=lr)
opt.zero_grad()
nn.CrossEntropyLoss()(net_a(xb), yb).backward()
opt.step()

# вариант B: наш цикл
for p in net_b.parameters():
    p.grad = None
nn.CrossEntropyLoss()(net_b(xb), yb).backward()
with torch.no_grad():
    for p in net_b.parameters():
        p -= lr * p.grad
        p.grad.zero_()

print("параметр      max |шаг optim.SGD - наш шаг|")
print("-" * 46)
for (na, pa), (nb, pb) in zip(net_a.named_parameters(), net_b.named_parameters()):
    print(f"{na:<12}  {(pa - pb).abs().max().item():.3e}")
print("\noptim.SGD с momentum=0 - это буквально p -= lr * p.grad")

### Что оптимизатор дает сверх этого: momentum

Momentum - это то, за чем к `optim` обращаются на практике. Вместо шага по
текущему градиенту шаг делается по его накопленному скользящему среднему:

$$v \leftarrow \mu v + g, \qquad p \leftarrow p - \eta v$$

где $\mu$ - коэффициент инерции (обычно 0.9). Эффект: градиентный шум по
батчам усредняется, а систематическое направление спуска усиливается -
примерно в $1/(1-\mu) = 10$ раз на длинном участке.

Реализуется в шесть строк, но мы его не писали, потому что он не нужен
для понимания обратного прохода. Ниже видно, что он дает.

In [ ]:
def iterate_minibatches(x, y, batch_size, shuffle=True, generator=None):
    total = x.shape[0]
    order = torch.randperm(total, generator=generator) if shuffle else torch.arange(total)
    for start in range(0, total, batch_size):
        idx = order[start:start + batch_size]
        yield x[idx], y[idx]


@torch.no_grad()
def evaluate(model, x, y, batch_size=1000):
    model.eval()
    loss_sum = correct = 0.0
    seen = 0
    crit = nn.CrossEntropyLoss()
    for xb_, yb_ in iterate_minibatches(x, y, batch_size, shuffle=False):
        z = model(xb_)
        k = yb_.shape[0]
        loss_sum += float(crit(z, yb_)) * k
        correct += float((z.argmax(dim=1) == yb_).float().mean()) * k
        seen += k
    model.train()
    return loss_sum / seen, correct / seen


def train_nn(momentum=0.0, epochs=10, lr=0.1, bs=64, seed=0):
    torch.manual_seed(seed)
    gen = torch.Generator().manual_seed(seed)
    model = MLP(d, n_hidden, num_classes)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.SGD(model.parameters(), lr=lr, momentum=momentum)

    accs, losses = [], []
    for epoch in range(1, epochs + 1):
        for xb_, yb_ in iterate_minibatches(x_train, y_train, bs, True, gen):
            optimizer.zero_grad()                     # = p.grad.zero_()
            loss = criterion(model(xb_), yb_)         # = cross_entropy(...)
            loss.backward()                           # то же самое
            optimizer.step()                          # = p -= lr * p.grad
            losses.append(float(loss.detach()))
        _, acc = evaluate(model, x_valid, y_valid)
        accs.append(acc)
        print(f"  эпоха {epoch:>2}/{epochs}: точность {acc * 100:.2f}%")
    return model, accs, losses


print("SGD без momentum (эквивалент ноутбука 03):")
net_plain, acc_plain, loss_plain = train_nn(momentum=0.0)

print("\nSGD с momentum=0.9:")
net_mom, acc_mom, loss_mom = train_nn(momentum=0.9)

---
## 5. Сводка по всем четырем ноутбукам

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.2))

ax = axes[0]
ax.axhline(92.43, ls="--", c="gray", lw=1.2,
           label="02: линейная вручную, 92.4%")
ax.axhline(97.42, ls="--", c="seagreen", lw=1.2,
           label="03: MLP вручную, 97.4%")
ax.plot(range(1, len(acc_plain) + 1), [a * 100 for a in acc_plain], "o-",
        color="steelblue", label=f"04: nn.Module, SGD, {acc_plain[-1] * 100:.2f}%")
ax.plot(range(1, len(acc_mom) + 1), [a * 100 for a in acc_mom], "s-",
        color="crimson", label=f"04: nn.Module, momentum=0.9, {acc_mom[-1] * 100:.2f}%")
ax.set_xlabel("эпоха"); ax.set_ylabel("точность на валидации, %")
ax.set_title("Точность: ручная реализация против штатной")
ax.legend(fontsize=9); ax.grid(alpha=0.3)

ax = axes[1]
window = 200
for data, label, color in [(loss_plain, "SGD", "steelblue"),
                           (loss_mom, "SGD + momentum 0.9", "crimson")]:
    sm = torch.tensor(data).unfold(0, window, 1).mean(dim=1)
    ax.plot(range(window - 1, len(data)), sm, color=color, lw=1.6, label=label)
ax.axhline(math.log(10), ls="--", c="gray", lw=1, label="log(10)")
ax.set_xlabel("шаг"); ax.set_ylabel("ошибка (скользящее среднее)")
ax.set_yscale("log")
ax.set_title("Ошибка на обучающих батчах"); ax.legend(); ax.grid(alpha=0.3)

plt.tight_layout(); plt.show()

print("Итоговая таблица:")
print(f"{'модель':<40} {'параметров':>12} {'точность':>10}")
print("-" * 65)
print(f"{'02: линейная, вручную':<40} {7850:>12} {'92.43%':>10}")
print(f"{'03: MLP 784-128-10, вручную':<40} {101770:>12} {'97.42%':>10}")
print(f"{'04: тот же MLP через nn.Module':<40} {101770:>12} "
      f"{acc_plain[-1] * 100:>9.2f}%")
print(f"{'04: тот же MLP + momentum 0.9':<40} {101770:>12} "
      f"{acc_mom[-1] * 100:>9.2f}%")

Ручная реализация и штатная дают **одну и ту же точность** - различия в
пределах шума SGD. Это и есть главный итог: `nn.Module` не содержит никакой
скрытой магии, которой у нас не было. Momentum дает заметную прибавку, и это
единственное содержательное улучшение в этом ноутбуке - но его мы тоже
разобрали формулой.

---
## 6. Использование модулей проекта

Рядом с ноутбуками в папке `src/mnist_manual/` лежат те же функции,
разложенные по модулям: `data`, `functional`, `grads`, `models`, `train`.
Это ровно тот код, который разобран в ноутбуках 02-03, без изменений -
удобно запускать из PyCharm и переиспользовать.

Ячейка ниже работает, если ноутбук запущен из папки проекта (локально).
В Colab она просто сообщит, что модулей нет, и это не помешает остальному
ноутбуку - он самодостаточен. Как подключить `src` в Colab, описано
в `README.md`.

In [ ]:
import sys

for candidate in ("src", "../src"):
    path = Path(candidate).resolve()
    if path.exists() and str(path) not in sys.path:
        sys.path.insert(0, str(path))

try:
    from mnist_manual import data as mdata
    from mnist_manual import grads as mgrads
    from mnist_manual.models import LinearModel, MLPModel
    from mnist_manual.train import evaluate as mevaluate
    from mnist_manual.train import train as mtrain

    print("модули проекта подключены\n")

    xt, yt, xv, yv = mdata.load_mnist()
    print(mdata.describe(xt, yt, "train"))

    gen = torch.Generator().manual_seed(0)
    model = MLPModel(generator=gen)
    print(f"\nMLPModel: {model.num_params()} параметров")

    # Один шаг со сверкой ручных градиентов - тот же код, что в ноутбуке 03
    xb_, yb_ = xt[:64], yt[:64]
    from mnist_manual import functional as mfunc

    loss = mfunc.cross_entropy(model(xb_), yb_)
    loss.backward()
    report = mgrads.compare_with_autograd(model.manual_grads(xb_, yb_),
                                          model.parameters())
    print()
    print(mgrads.format_report(report))

except ImportError as e:
    print("модули проекта недоступны (это ожидаемо в Colab):", e)
    print("см. README.md, раздел про запуск в Colab")

---
## Итоги проекта

**Что было сделано.** MNIST решен дважды с нуля на тензорных операциях:
линейной моделью (92.4%) и сетью с одним скрытым слоем (97.4%). Все
градиенты выведены на бумаге, выписаны формулами и проверены против
autograd и против конечных разностей на каждом этапе обучения.

**Формулы, которые стоит помнить:**

| Что | Формула |
|---|---|
| softmax | $p_k = e^{z_k - m} / \sum_j e^{z_j - m}$, $m = \max_j z_j$ |
| перекрестная энтропия | $L = -\frac{1}{B}\sum_n \log p_{n, y_n}$ |
| градиент по логитам | $\partial L / \partial Z = (\mathrm{softmax}(Z) - Y)/B$ |
| линейный слой | $\partial L/\partial W = X^\top G$, $\ \partial L/\partial b = \mathrm{sum}(G,0)$, $\ \partial L/\partial X = G W^\top$ |
| ReLU | $\partial L/\partial Z = (\partial L/\partial H) \odot [Z > 0]$ |
| шаг SGD | $p \leftarrow p - \eta\, \partial L/\partial p$ |

**Приемы отладки, которые переносятся на любую задачу:**

1. прикидка стартовой ошибки: $\log(\text{число классов})$ - если не сходится,
   дальше запускать бессмысленно;
2. контроль порядка множителей в backward по формам тензоров;
3. тройная сверка градиента: аналитика, конечные разности в `float64`,
   autograd;
4. матрица ошибок вместо одного процента точности;
5. проверка, что `.grad` обнуляется, а шаг идет под `no_grad()`.

**Что дальше, если захочется продолжить:**

* сверточная сеть - учитывает двумерную структуру картинки, на MNIST дает
  99%+; потребует развернуть обратный проход через свертку;
* dropout и регуляризация - бороться с переобучением на более сложных данных;
* Adam вместо SGD - адаптивный шаг для каждого параметра;
* нормализация батча - стабилизирует обучение глубоких сетей.